#Transformation logic

In [0]:
query= '''
SELECT 
    ROW_NUMBER() OVER (ORDER BY ci.customer_id) AS customer_number,
    ci.customer_id,
    ci.customer_key,
    ci.customer_firstname,
    ci.customer_lastname,
    ci.customer_marital_status,
    COALESCE(NULLIF(ci.customer_gender, 'n/a'), ca.gender, 'n/a') AS gender,
    ci.customer_created_date,
    ca.birth_date,
    la.country

FROM 
    databricks_lakehouse.silver.crm_customers ci
LEFT JOIN databricks_lakehouse.silver.erp_customers ca
ON ca.customer_key = ci.customer_key
LEFT JOIN databricks_lakehouse.silver.erp_customer_location la
ON la.customer_key = ci.customer_key

'''
df = spark.sql(query)
df.display()

#Writing in gold

In [0]:
(
    df.write.mode("overwrite")
    .format("delta")
    .saveAsTable("databricks_lakehouse.gold.dim_customers")
)

#Checiking the table

In [0]:
%sql
select * from databricks_lakehouse.gold.dim_customers limit 5